In [5]:
import sqlite3
import pandas as pd
import networkx as nx
import os

# 1. Data Extraction
db_path = '../data/document_analysis.db'
conn = sqlite3.connect(db_path)

query = """
SELECT 
    COALESCE(ea1.canonical_name, t.actor) AS source,
    COALESCE(ea2.canonical_name, t.target) AS target
FROM rdf_triples t
LEFT JOIN entity_aliases ea1 ON t.actor = ea1.original_name
LEFT JOIN entity_aliases ea2 ON t.target = ea2.original_name
"""
df_edges = pd.read_sql_query(query, conn)
conn.close()

df_clean = df_edges.dropna(subset=['source', 'target'])
df_clean = df_clean[df_clean['source'] != df_clean['target']].copy()

edge_weights = df_clean.groupby(['source', 'target']).size().reset_index(name='weight')

In [6]:
# 2. Network Construction and Metrics Calculation (RQ2)
G = nx.from_pandas_edgelist(edge_weights, 'source', 'target', ['weight'])

degree_dict = nx.degree_centrality(G)
betweenness_dict = nx.betweenness_centrality(G, k=100, weight='weight', seed=42) 

df_metrics = pd.DataFrame({
    'Node': list(G.nodes()),
    'Degree_Centrality': list(degree_dict.values()),
    'Betweenness_Centrality': list(betweenness_dict.values())
}).sort_values(by='Betweenness_Centrality', ascending=False)

In [7]:
# 3. Data Export
output_dir = '../data/'
os.makedirs(output_dir, exist_ok=True)

df_metrics.to_csv(os.path.join(output_dir, 'nodes_base_metrics.csv'), index=False)
edge_weights.to_csv(os.path.join(output_dir, 'edges_base.csv'), index=False)

nx.set_node_attributes(G, degree_dict, 'degree')
nx.set_node_attributes(G, betweenness_dict, 'betweenness')
nx.write_graphml(G, os.path.join(output_dir, 'network_investigativa.graphml'))

# Rendering nativo di Jupyter per i Top 10 Broker
df_metrics.head(10)

,Node,Degree_Centrality,Betweenness_Centrality
113,unknown person A,0.069647,0.177799
15,Jeffrey Epstein,0.103110,0.152678
188,Donald Trump,0.029041,0.077484
117,Alan M. Dershowitz,0.014663,0.035585
1139,Barack Obama,0.006893,0.029428
460,United States,0.006428,0.025349
4033,Author,0.004995,0.022027
265,unknown person B,0.008666,0.021170
523,Bill Clinton,0.005479,0.016592
519,Ehud Barak,0.012175,0.015461
